In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing #this loads the dataset provided through scikit-learn
from sklearn.model_selection import train_test_split #divide data into training and testing
from sklearn.impute import SimpleImputer #systematic way to replace missing values.
from sklearn.preprocessing import StandardScaler #transform features so that they are approximately on a comparable scale
from sklearn.linear_model import LinearRegression #an ML model
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score) #evaluation metrics
#here b1,b2,b3...,bn are called coefficients - it tells how much predicted target changes when the correspending input feature change by one unit, while other features are kept constant.

In [5]:
#Loading Data
housing = fetch_california_housing(as_frame = True)
df = housing.frame
print("Dataset shape: ", df.shape)
df.head()

Dataset shape:  (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [6]:
# Knowing about the dataset
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nStatistical summary:")
df.describe()


Data types:
MedInc         float64
HouseAge       float64
AveRooms       float64
AveBedrms      float64
Population     float64
AveOccup       float64
Latitude       float64
Longitude      float64
MedHouseVal    float64
dtype: object

Missing values:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Duplicate rows:
0

Statistical summary:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [7]:
#creating a dirty dataset
df_dirty = df.copy()

#put a missing value into MedInc
df_dirty.loc[10, "MedInc"] = np.nan

#put a missing value into HouseAge
df_dirty.loc[20, "HouseAge"] = np.nan

#put a missing value into AveRooms
df_dirty.loc[30, "AveRooms"] = np.nan

#put a missing value into Population
df_dirty.loc[40, "Population"] = np.nan
print(df_dirty.isnull().sum())

MedInc         1
HouseAge       1
AveRooms       1
AveBedrms      0
Population     1
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


In [10]:
#X contains the input features.
#these are the variables that the model will use to predict the house value.
X = df_dirty[
    [
      "MedInc",
      "HouseAge",
      "AveRooms",
      "AveBedrms",
      "Population",
      "AveOccup",
      "Latitude",
      "Longitude"
   ]
]

#y contains the output/target variable.
#this is the value that we want model to predict
y = df_dirty["MedHouseVal"]

#display the input features columns
print("\nInput Features (X):")
print(X.columns)

#display the target column.
print("\nTarget (y):")
print(y.name)

#display the shape of X.
print("\nShape of X:")
print(X.shape)

#dislay the shape of y.
print("\nShape of y:")
print(y.shape)


Input Features (X):
Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object')

Target (y):
MedHouseVal

Shape of X:
(20640, 8)

Shape of y:
(20640,)


In [12]:
#80% of the data will be used for training.
#20% of the data will be used for testing.
#random_state = 42 makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state = 42
)

#display training data size.
print("\nTraining Data Shape:")
print(X_train.shape)

#display testing data size.
print("\nTesting Data Shape:")
print(X_test.shape)


Training Data Shape:
(16512, 8)

Testing Data Shape:
(4128, 8)


In [13]:
#median imputation means missing values will be replaced by the median value of the feature.
imputer = SimpleImputer(strategy = "median")

#learn the median values only from training data.
#we should not clac these values using test data, otherwise information from the test set leak into training.
X_train_imputed = imputer.fit_transform(X_train)

#use the already learned training medians to fill missing values in the test data.
X_test_imputed = imputer.transform(X_test)

In [14]:
#standard scaler converts features approximately to:
#mean = 0
#standard deviation = 1

#formula:
# z = (x - mean) / standard_deviation

scaler = StandardScaler()

#calc scaling parameters from training data and then transforms the training data.
X_train_scaled = scaler.fit_transform(X_train_imputed)

#apply the same scaling parameters to test data.
X_test_scaled = scaler.transform(X_test_imputed)

In [16]:
#create a linear regression model.
model = LinearRegression()

#the model learns the relationship between: X_train_scaled -> y_train (y = b0 + b1X1 + b2X2 + ..... + b8X8)
model.fit(X_train_scaled, y_train)

LinearRegression()

In [17]:
#intercept_ represents b0 in the equation.
print("\nIntercept (b0):")
print(model.intercept_)


Intercept (b0):
2.0719469373788777
